# Clase 223 — Tipos de sesgo algorítmico: diagnóstico con dataset sintético

Reproducimos los 6 tipos del framework Suresh-Guttag (2021) sobre un dataset de préstamos sintético. Requiere: `pip install scikit-learn pandas numpy`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

rng = np.random.default_rng(42)
N = 5000

grupo = rng.choice(['A', 'B'], size=N, p=[0.7, 0.3])
ingreso = np.where(grupo == 'A', rng.normal(50, 15, N), rng.normal(35, 12, N))
score_credito = np.where(grupo == 'A', rng.normal(700, 50, N), rng.normal(620, 60, N))
zip_premium = np.where(grupo == 'A', rng.binomial(1, 0.6, N), rng.binomial(1, 0.2, N))

capacidad_real = (ingreso > 40).astype(int)
sesgo_historico = np.where(grupo == 'A', 0.25, -0.25)
p_aprobado = np.clip(0.5 + 0.3 * (capacidad_real - 0.5) + sesgo_historico, 0.05, 0.95)
y = rng.binomial(1, p_aprobado)

df = pd.DataFrame({'grupo': grupo, 'ingreso': ingreso, 'score': score_credito,
                   'zip_premium': zip_premium, 'capacidad_real': capacidad_real, 'y': y})
print(df.groupby('grupo')['y'].mean().round(3))

## 1. Sesgo histórico: el modelo lo reproduce aunque saquemos la variable sensible

In [ ]:
X = df[['ingreso', 'score', 'zip_premium']].values
y_arr = df['y'].values
g = df['grupo'].values

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(X, y_arr, g, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred = model.predict(X_te)

sr_A = y_pred[g_te == 'A'].mean()
sr_B = y_pred[g_te == 'B'].mean()
print(f'Selection rate grupo A: {sr_A:.3f}')
print(f'Selection rate grupo B: {sr_B:.3f}')
print(f'Ratio B/A: {sr_B / max(sr_A, 1e-6):.2f}  → ratio < 0.8 viola regla 80% (EEOC)')

El modelo **nunca vio** `grupo` pero reprodujo el gap vía proxies (`ingreso`, `score`, `zip_premium`). Fairness through unawareness no alcanza.

## 2. Sesgo de representación (patrón Gender Shades)

Re-sampleamos para que el grupo B sea apenas 10% del train — el patrón que Buolamwini & Gebru documentaron en IJB-A.

In [ ]:
df_A = df[df.grupo == 'A'].sample(2000, random_state=42)
df_B = df[df.grupo == 'B'].sample(220, random_state=42)
df_repr = pd.concat([df_A, df_B]).sample(frac=1, random_state=42)

X_repr = df_repr[['ingreso', 'score', 'zip_premium']].values
y_repr = df_repr['y'].values
g_repr = df_repr['grupo'].values

X_trR, X_teR, y_trR, y_teR, g_trR, g_teR = train_test_split(
    X_repr, y_repr, g_repr, test_size=0.3, random_state=42, stratify=g_repr)
m2 = LogisticRegression(max_iter=1000).fit(X_trR, y_trR)
p2 = m2.predict(X_teR)

acc_global = accuracy_score(y_teR, p2)
acc_A = accuracy_score(y_teR[g_teR == 'A'], p2[g_teR == 'A'])
acc_B = accuracy_score(y_teR[g_teR == 'B'], p2[g_teR == 'B'])
print(f'Accuracy global:   {acc_global:.3f}  ← el número que se reporta')
print(f'Accuracy grupo A:  {acc_A:.3f}')
print(f'Accuracy grupo B:  {acc_B:.3f}  ← el número que NO se reporta')
print(f'Gap A-B:           {acc_A - acc_B:.3f}')

## 3. Sesgo de medición: el proxy ≠ el target real

Entrenamos sobre `y_proxy` (lo que medimos, ej. re-arresto) con ruido correlacionado al grupo, mientras `capacidad_real` es el target justo. El modelo aprende el ruido.

In [ ]:
ruido_grupal = np.where(df['grupo'] == 'B', rng.binomial(1, 0.3, N), 0)
y_proxy = np.clip(df['capacidad_real'].values + ruido_grupal - rng.binomial(1, 0.05, N), 0, 1)

X_full = df[['ingreso', 'score', 'zip_premium']].values
y_true = df['capacidad_real'].values

m_proxy = LogisticRegression(max_iter=1000).fit(X_full, y_proxy)
m_true  = LogisticRegression(max_iter=1000).fit(X_full, y_true)

for gname in ['A', 'B']:
    mask = df.grupo.values == gname
    auc_p = roc_auc_score(y_true[mask], m_proxy.predict_proba(X_full[mask])[:, 1])
    auc_t = roc_auc_score(y_true[mask], m_true.predict_proba(X_full[mask])[:, 1])
    print(f'Grupo {gname}: AUC train-en-proxy={auc_p:.3f} | AUC train-en-truth={auc_t:.3f}')

El modelo entrenado en `y_proxy` evalúa peor (contra la verdad) en el grupo donde el proxy tenía ruido.

## 4. Sesgo de agregación: un modelo único vs un modelo por subgrupo (Simpson)

In [ ]:
score_sim = rng.normal(0, 1, N)
logit = np.where(df.grupo.values == 'A', 2*score_sim, -2*score_sim)
y_sim = rng.binomial(1, 1/(1 + np.exp(-logit)))
X_sim = score_sim.reshape(-1, 1)

m_unico = LogisticRegression().fit(X_sim, y_sim)
auc_unico_A = roc_auc_score(y_sim[df.grupo == 'A'], m_unico.predict_proba(X_sim[df.grupo == 'A'])[:, 1])
auc_unico_B = roc_auc_score(y_sim[df.grupo == 'B'], m_unico.predict_proba(X_sim[df.grupo == 'B'])[:, 1])

mA = LogisticRegression().fit(X_sim[df.grupo == 'A'], y_sim[df.grupo == 'A'])
mB = LogisticRegression().fit(X_sim[df.grupo == 'B'], y_sim[df.grupo == 'B'])
auc_sub_A = roc_auc_score(y_sim[df.grupo == 'A'], mA.predict_proba(X_sim[df.grupo == 'A'])[:, 1])
auc_sub_B = roc_auc_score(y_sim[df.grupo == 'B'], mB.predict_proba(X_sim[df.grupo == 'B'])[:, 1])

print(f'Modelo único        → AUC A={auc_unico_A:.3f}  AUC B={auc_unico_B:.3f}')
print(f'Modelo por subgrupo → AUC A={auc_sub_A:.3f}  AUC B={auc_sub_B:.3f}')
print('\n→ Simpson: la relación score→y es opuesta por grupo. El modelo único promedia y pierde.')

## 5. Sesgo de evaluación + despliegue

Test set 90% A, pero deployment será 50/50. La métrica de test sobre-estima la performance real.

In [ ]:
idx_A = np.where(g_teR == 'A')[0]
idx_B = np.where(g_teR == 'B')[0]

sample_A_skewed = rng.choice(idx_A, size=min(180, len(idx_A)), replace=False)
sample_B_skewed = rng.choice(idx_B, size=min(20, len(idx_B)), replace=False)
idx_skewed = np.concatenate([sample_A_skewed, sample_B_skewed])

n_per = min(len(idx_A), len(idx_B), 100)
sample_A_real = rng.choice(idx_A, size=n_per, replace=False)
sample_B_real = rng.choice(idx_B, size=n_per, replace=False)
idx_real = np.concatenate([sample_A_real, sample_B_real])

acc_paper  = accuracy_score(y_teR[idx_skewed], p2[idx_skewed])
acc_deploy = accuracy_score(y_teR[idx_real],   p2[idx_real])
print(f'Accuracy en test sesgado (lo reportado):       {acc_paper:.3f}')
print(f'Accuracy en deployment realista (lo que ocurre): {acc_deploy:.3f}')
print(f'Brecha evaluación-despliegue: {acc_paper - acc_deploy:+.3f}')

## Ejercicio guiado

1. Cargá UCI Adult (`sex`, `race`). Repetí el análisis de los 6 tipos sobre datos reales.
2. Variá el porcentaje del grupo minoritario (50%, 20%, 5%, 1%) y graficá `accuracy_minor` vs `%minor` — confirmá la curva de Gender Shades.
3. En el caso del proxy (paso 3), proponé una corrección: ¿podés re-pesar muestras para compensar el ruido conocido?
4. Implementá la **regla 80%** (EEOC): función que dado `y_pred` y `grupo` devuelve `(ratio, passes_bool)`.
5. Escribí 1 párrafo aplicando Suresh-Guttag a un modelo real (en el trabajo, en una clase previa, en la prensa).

## Conclusiones

- El sesgo no es un bug del modelo: nace en alguna fase del ML life cycle. Diagnosticá **dónde** antes de mitigar.
- *Fairness through unawareness* (sacar la variable sensible) **no funciona** — los proxies persisten.
- Reportá métricas **stratificadas por subgrupo**, siempre. La accuracy global esconde Gender Shades.
- Sesgo de medición y histórico requieren **decisiones humanas** sobre el target; ningún algoritmo lo resuelve.
- Simpson: un modelo único puede ser peor que k modelos por subgrupo si las distribuciones difieren.
- Test ≠ deployment: auditá la representatividad del test antes de creer en su accuracy.